In [1]:
# Efficient ResNet baseline: original implementation, trained from scratch.
# Residual learning reference: He et al., https://arxiv.org/abs/1512.03385
from pathlib import Path
import random, json, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.model_selection import train_test_split
import kagglehub

SEED, EPOCHS, BATCH = 42, 30, 256
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
assert torch.cuda.is_available(), 'Select Settings > Accelerator > GPU before running.'
torch.backends.cudnn.benchmark = True
torch.set_num_threads(4)
device = torch.device('cuda')
DATA = Path(kagglehub.competition_download('cifar-10-res-net-architecture-challenge-dl'))
OUT = Path('/kaggle/working'); OUT.mkdir(exist_ok=True)
train = pd.read_csv(DATA/'train_labels.csv')
test = pd.read_csv(DATA/'test.csv')
sample = pd.read_csv(DATA/'sample_submission.csv')
assert len(train)==50000 and train.label.between(0,9).all()
assert train.id.is_unique and test.id.is_unique and sample.id.is_unique
print('GPU:', torch.cuda.get_device_name(0), '| Data:', DATA, flush=True)

def load_images(frame, folder):
    def read(name):
        with Image.open(DATA/folder/name) as im:
            a = np.array(im.convert('RGB'))
        assert a.shape == (32,32,3)
        return a
    with ThreadPoolExecutor(max_workers=8) as pool:
        return np.stack(list(pool.map(read, frame.filename)))
print('Loading permitted images into memory...', flush=True)
images = load_images(train, 'train_images')
test_images = load_images(test, 'test_images')
labels = train.label.to_numpy(dtype=np.int64)
tr, va = train_test_split(np.arange(len(train)), test_size=5000, stratify=labels, random_state=SEED)
np.savez(OUT/'split_seed42.npz', train=tr, validation=va)
mean, std = (0.4914,0.4822,0.4465), (0.2470,0.2435,0.2616)
aug = T.Compose([T.RandomCrop(32,padding=4), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(mean,std)])
plain = T.Compose([T.ToTensor(), T.Normalize(mean,std)])
class Images(Dataset):
    def __init__(self, x, y=None, transform=plain):
        self.x,self.y,self.transform=x,y,transform
    def __len__(self): return len(self.x)
    def __getitem__(self,i):
        x=self.transform(Image.fromarray(self.x[i]))
        return x if self.y is None else (x,int(self.y[i]))
def loader(ds, shuffle=False):
    return DataLoader(ds,batch_size=BATCH,shuffle=shuffle,num_workers=2,pin_memory=True,persistent_workers=True)
train_dl=loader(Images(images[tr],labels[tr],aug),True)
val_dl=loader(Images(images[va],labels[va]))
test_dl=loader(Images(test_images))
class Block(nn.Module):
    def __init__(self,cin,cout,stride=1):
        super().__init__()
        self.f=nn.Sequential(nn.Conv2d(cin,cout,3,stride,1,bias=False),nn.BatchNorm2d(cout),nn.ReLU(inplace=True),nn.Conv2d(cout,cout,3,1,1,bias=False),nn.BatchNorm2d(cout))
        self.skip=nn.Identity() if cin==cout and stride==1 else nn.Sequential(nn.Conv2d(cin,cout,1,stride,bias=False),nn.BatchNorm2d(cout))
        self.relu=nn.ReLU(inplace=True)
    def forward(self,x): return self.relu(self.f(x)+self.skip(x))
class SmallResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem=nn.Sequential(nn.Conv2d(3,32,3,1,1,bias=False),nn.BatchNorm2d(32),nn.ReLU(inplace=True))
        layers=[]; cin=32
        for stage,cout in enumerate([32,64,128,256]):
            layers += [Block(cin,cout,1 if stage==0 else 2),Block(cout,cout)]
            cin=cout
        self.body=nn.Sequential(*layers)
        self.pool=nn.AdaptiveAvgPool2d(1); self.fc=nn.Linear(256,10)
    def forward(self,x): return self.fc(self.pool(self.body(self.stem(x))).flatten(1))
model=SmallResNet().to(device)
params=sum(p.numel() for p in model.parameters() if p.requires_grad)
assert params<=5_000_000, params
print('Trainable parameters:',params,flush=True)
config=dict(seed=SEED,epochs=EPOCHS,batch_size=BATCH,parameters=params,optimizer='SGD',lr=0.1,momentum=0.9,weight_decay=5e-4,label_smoothing=0.1,blocks=[2,2,2,2],widths=[32,64,128,256],pretrained=False)
(OUT/'config.json').write_text(json.dumps(config,indent=2))
optimizer=torch.optim.SGD(model.parameters(),lr=0.1,momentum=0.9,weight_decay=5e-4,nesterov=True)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)
criterion=nn.CrossEntropyLoss(label_smoothing=0.1)
scaler=torch.amp.GradScaler('cuda')
best=-1.; history=[]
for epoch in range(EPOCHS):
    started=time.time(); model.train(); total_loss=0.; correct=total=0
    for x,y in train_dl:
        x,y=x.to(device,non_blocking=True),y.to(device,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=torch.float16):
            logits=model(x); loss=criterion(logits,y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        total_loss+=loss.item()*len(y); correct+=(logits.argmax(1)==y).sum().item(); total+=len(y)
    model.eval(); vc=vn=0
    with torch.inference_mode():
        for x,y in val_dl:
            x,y=x.to(device),y.to(device)
            with torch.autocast('cuda',dtype=torch.float16): pred=model(x)
            vc+=(pred.argmax(1)==y).sum().item(); vn+=len(y)
    acc=vc/vn
    row=dict(epoch=epoch+1,loss=total_loss/total,train_accuracy=correct/total,val_accuracy=acc,seconds=time.time()-started,lr=optimizer.param_groups[0]['lr'])
    history.append(row); pd.DataFrame(history).to_csv(OUT/'training_log.csv',index=False)
    if acc>best:
        best=acc; torch.save(dict(model=model.state_dict(),config=config,val_accuracy=best),OUT/'best.pt')
    scheduler.step()
    torch.save(dict(model=model.state_dict(),optimizer=optimizer.state_dict(),scheduler=scheduler.state_dict(),scaler=scaler.state_dict(),epoch=epoch+1,config=config),OUT/'last.pt')
    print(f'Epoch {epoch+1}/{EPOCHS}: loss={row["loss"]:.4f} train={correct/total:.4f} val={acc:.4f} best={best:.4f} seconds={row["seconds"]:.0f}',flush=True)
checkpoint=torch.load(OUT/'best.pt',map_location=device,weights_only=False)
model.load_state_dict(checkpoint['model']); model.eval(); predictions=[]
with torch.inference_mode():
    for x in test_dl:
        x=x.to(device)
        with torch.autocast('cuda',dtype=torch.float16):
            scores=(model(x)+model(torch.flip(x,[3])))/2
        predictions.extend(scores.argmax(1).cpu().tolist())
lookup=pd.Series(predictions,index=test.id)
submission=sample[['id']].copy(); submission['label']=submission.id.map(lookup)
assert len(submission)==len(test) and submission.label.notna().all()
submission.label=submission.label.astype(int)
assert submission.columns.tolist()==['id','label'] and submission.label.between(0,9).all()
assert submission.id.tolist()==sample.id.tolist()
submission.to_csv(OUT/'submission.csv',index=False)
print('Saved submission.csv:',submission.shape,'Best validation:',best,flush=True)
print(submission.head())


GPU: Tesla T4 | Data: /kaggle/input/competitions/cifar-10-res-net-architecture-challenge-dl
Loading permitted images into memory...
Trainable parameters: 2797610
Epoch 1/30: loss=1.9401 train=0.3324 val=0.4548 best=0.4548 seconds=35
Epoch 2/30: loss=1.5408 train=0.5169 val=0.5678 best=0.5678 seconds=10
Epoch 3/30: loss=1.3303 train=0.6252 val=0.6590 best=0.6590 seconds=10
Epoch 4/30: loss=1.1804 train=0.6968 val=0.6710 best=0.6710 seconds=10
Epoch 5/30: loss=1.0694 train=0.7501 val=0.7042 best=0.7042 seconds=10
Epoch 6/30: loss=0.9968 train=0.7850 val=0.7474 best=0.7474 seconds=10
Epoch 7/30: loss=0.9430 train=0.8078 val=0.7500 best=0.7500 seconds=10
Epoch 8/30: loss=0.9066 train=0.8238 val=0.8210 best=0.8210 seconds=10
Epoch 9/30: loss=0.8749 train=0.8386 val=0.8216 best=0.8216 seconds=10
Epoch 10/30: loss=0.8436 train=0.8508 val=0.7974 best=0.8216 seconds=10
Epoch 11/30: loss=0.8223 train=0.8630 val=0.8356 best=0.8356 seconds=10
Epoch 12/30: loss=0.7998 train=0.8712 val=0.8398 best=0